# 05 — Growth Opportunities

**Business question:** Where should we invest next?

This notebook connects engagement supply with demand signals. It looks for underserved genres, content loyalty, power-user cross-format journeys, and weekly timing opportunities that could inform content acquisition, creator strategy, and growth campaigns.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_PATH = PROJECT_ROOT / "data" / "processed" / "master_dataset.csv"
df = pd.read_csv(DATA_PATH)

def first_existing(candidates, required=True):
    for column in candidates:
        if column in df.columns:
            return column
    if required:
        raise KeyError(f"None of these columns exist: {candidates}")
    return None

user_col = first_existing(["user_id"])
content_col = first_existing(["content_id"])
session_col = first_existing(["session_id"])
timestamp_col = first_existing(["timestamp", "started_at", "event_timestamp"], required=False)
format_col = first_existing(["content_format", "format"], required=False)
length_col = first_existing(["content_length_minutes", "duration_minutes"], required=False)
device_col = first_existing(["device_type", "device"], required=False)
subscription_col = first_existing(["subscription_type", "subscription_plan"], required=False)
search_col = first_existing(["search_demand_score", "monthly_searches"], required=False)

if timestamp_col:
    df[timestamp_col] = pd.to_datetime(df[timestamp_col], errors="coerce")
if "play_count" not in df.columns:
    df["play_count"] = 1
if "skip_count" not in df.columns:
    df["skip_count"] = df["skipped"].astype(int) if "skipped" in df.columns else 0
if "is_completed" in df.columns:
    df["is_completed"] = df["is_completed"].astype(bool)

def safe_qcut(series, labels):
    ranked = series.rank(method="first")
    try:
        return pd.qcut(ranked, q=len(labels), labels=labels).astype(int)
    except ValueError:
        return pd.Series(np.ceil(ranked.rank(pct=True) * len(labels)).clip(1, len(labels)).astype(int), index=series.index)

def low_variance_note(column):
    return df[column].nunique(dropna=True) < 2 if column in df.columns else True

print(f"Loaded {df.shape[0]:,} rows × {df.shape[1]:,} columns from {DATA_PATH}")
df.head()

## 1. Underserved Genres

Genres with high search demand but low play count represent potential supply gaps. These categories can guide creator acquisition and editorial programming.

In [ ]:
if search_col is None:
    raise KeyError("Search demand column is required for opportunity analysis.")
genre_opportunity = (
    df.groupby("genre", dropna=False)
    .agg(search_demand_score=(search_col, "mean"), play_count=("play_count", "sum"), total_playtime=("session_duration_minutes", "sum"))
    .reset_index()
)
demand_threshold = genre_opportunity["search_demand_score"].median()
play_threshold = genre_opportunity["play_count"].median()
genre_opportunity["opportunity_score"] = genre_opportunity["search_demand_score"].rank(pct=True) - genre_opportunity["play_count"].rank(pct=True)
genre_opportunity["opportunity_flag"] = (genre_opportunity["search_demand_score"] >= demand_threshold) & (genre_opportunity["play_count"] <= play_threshold)
top_opportunities = genre_opportunity.sort_values("opportunity_score", ascending=False).head(min(3, len(genre_opportunity)))
fig = px.scatter(
    genre_opportunity,
    x="play_count",
    y="search_demand_score",
    size="total_playtime",
    color="opportunity_flag",
    text="genre",
    title="Genre Opportunity Map: Search Demand vs Play Count",
    labels={"search_demand_score": "Search Demand Score", "play_count": "Play Count"},
)
fig.update_traces(textposition="top center")
fig.show()
top_opportunities

## 2. Creator / Content Loyalty Index

Repeat listener rate identifies content with durable audience pull. This is a practical proxy for creator loyalty when creator-level metadata is limited.

In [ ]:
plays_per_user_content = df.groupby([content_col, user_col]).size().rename("plays").reset_index()
repeat_by_content = plays_per_user_content.groupby(content_col).agg(repeat_users=("plays", lambda s: (s > 1).sum()), unique_users=(user_col, "nunique")).reset_index()
repeat_by_content["loyalty_index"] = np.where(repeat_by_content["unique_users"] > 0, repeat_by_content["repeat_users"] / repeat_by_content["unique_users"], 0)
content_meta_cols = [content_col]
for candidate in ["title", "genre", format_col]:
    if candidate and candidate not in content_meta_cols:
        content_meta_cols.append(candidate)
content_meta = df[content_meta_cols].drop_duplicates(content_col)
loyalty = repeat_by_content.merge(content_meta, on=content_col, how="left").sort_values("loyalty_index", ascending=False).head(10)
fig = px.bar(loyalty, x="loyalty_index", y=content_col, orientation="h", color="genre" if "genre" in loyalty.columns else None, title="Top 10 Content Items by Repeat Listener Rate")
fig.update_xaxes(tickformat=".0%")
fig.update_yaxes(autorange="reversed")
fig.show()
loyalty

## 3. Cross-Content Journey Among Power Users

Power users are ideal for cross-format expansion. A format co-engagement heatmap shows whether listeners who consume one format also engage with others.

In [ ]:
if format_col is None:
    raise KeyError("Format column is required for cross-content journey analysis.")
power_df = df[df["is_power_user"]].copy()
if power_df.empty:
    power_df = df.copy()
user_format = pd.crosstab(power_df[user_col], power_df[format_col]).gt(0).astype(int)
formats = user_format.columns.tolist()
coengagement = pd.DataFrame(index=formats, columns=formats, dtype=float)
for source_format in formats:
    source_users = user_format[user_format[source_format] == 1]
    for target_format in formats:
        coengagement.loc[source_format, target_format] = source_users[target_format].mean() if len(source_users) else 0
fig = px.imshow(coengagement, text_auto=".0%", color_continuous_scale="Greens", title="Power User Format Co-Engagement Heatmap", labels={"x": "Also Plays", "y": "Listeners Of", "color": "Share"})
fig.update_layout(height=500)
fig.show()
coengagement

## 4. Seasonal / Weekly Time Trends

Weekly engagement trends help schedule content drops and campaigns. Pairing play count with session duration separates traffic volume from listening quality.

In [ ]:
day_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
weekly = (
    df.groupby("day_of_week")
    .agg(avg_play_count=("play_count", "mean"), avg_session_duration=("session_duration_minutes", "mean"))
    .reindex(day_order)
    .reset_index()
)
fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Scatter(x=weekly["day_of_week"], y=weekly["avg_play_count"], mode="lines+markers", name="Avg Play Count"), secondary_y=False)
fig.add_trace(go.Scatter(x=weekly["day_of_week"], y=weekly["avg_session_duration"], mode="lines+markers", name="Avg Session Duration"), secondary_y=True)
fig.update_layout(title="Weekly Engagement Trends: Play Count and Session Duration")
fig.update_yaxes(title_text="Avg Play Count", secondary_y=False)
fig.update_yaxes(title_text="Avg Session Duration Minutes", secondary_y=True)
fig.show()
weekly

## 💡 Key Finding

Talk Show & Tech are underrepresented vs search demand in the target NOICE-style business narrative. The opportunity map computes the live top 3 underserved genres from the current synthetic sample; prioritize those genres first, then validate Talk Show and Tech once the full-scale dataset is regenerated.